<a href="https://colab.research.google.com/github/merymery011/Bird-detection-Darjaoui-Meryem/blob/main/bird_detectionDarjaoui_Meryem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install segmentation-models
!pip install efficientnet==1.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: efficientnet
    Found existing installation: efficientnet 1.0.0
    Uninstalling efficientnet-1.0.0:
      Successfully uninstalled efficientnet-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
segmentation-models 1.0.1 requires efficientnet==1.0.0, but you have efficientnet 1.1.1 which is incompatible.


In [2]:
import cv2
import numpy as np
import segmentation_models as sm
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
from google.colab.patches import cv2_imshow

# Configuration du backend
sm.set_framework('tf.keras')
sm.framework()

# Charger U-Net avec ResNet34 pré-entraîné
model = sm.Unet('resnet34', encoder_weights='imagenet', classes=1, activation='sigmoid')

# Charger les poids personnalisés si tu en as (optionnel)
# model.load_weights('mon_modele_unet_resnet34.h5')

# Paramètres vidéo
video_path = 'Birds-1.mp4'
cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter('Tracked_Birds_UNET.avi', cv2.VideoWriter_fourcc(*'XVID'), fps, (frame_width, frame_height))


kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

def get_unet_mask(frame):
    resized = cv2.resize(frame, (256, 256))
    arr = img_to_array(resized) / 255.0
    arr = np.expand_dims(arr, axis=0)
    pred = model.predict(arr)[0, :, :, 0]
    pred = cv2.resize(pred, (frame.shape[1], frame.shape[0]))
    pred_mask = (pred > 0.5).astype(np.uint8) * 255
    return pred_mask

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 60, 255, cv2.THRESH_BINARY_INV)
    morph_mask = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_open)
    morph_mask = cv2.dilate(morph_mask, kernel_dilate, iterations=1)


    unet_mask = get_unet_mask(frame)


    combined_mask = np.maximum(morph_mask, unet_mask)


    contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = cv2.contourArea(cnt)
        aspect_ratio = w / h if h != 0 else 0

        if area > 1000 and 0.5 < aspect_ratio < 2.0:
            center = (x + w // 2, y + h // 2)
            radius = int(max(w, h) / 2)
            cv2.circle(frame, center, radius, (0, 0, 255), 2)
            line_length = 10
            cv2.line(frame, (center[0] - line_length, center[1]), (center[0] + line_length, center[1]), (0, 0, 255), 2)
            cv2.line(frame, (center[0], center[1] - line_length), (center[0], center[1] + line_length), (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()
print("Tracking terminé avec U-Net !")

Segmentation Models: using `keras` framework.
85521592/85521592 ━━━━━━━━━━━━━━━━━━━━ 60s 1us/step
Tracking terminé avec U-Net !


In [3]:
# 2. Importer les bibliothèques
import cv2
import os
from matplotlib import pyplot as plt
from google.colab import files

# 3. Créer un dossier pour sauvegarder les frames
os.makedirs("Frames-Fish", exist_ok=True)

# 4. Upload vidéo manuellement (si pas encore fait)
uploaded = files.upload()  # Choisis un fichier .mp4

# 5. Lire la vidéo (remplacer le nom du fichier par celui que tu as uploadé)
video_path = list(uploaded.keys())[0]
cap = cv2.VideoCapture(video_path)

Saving Birds-2 (1).mp4 to Birds-2 (1).mp4


In [4]:
# 6. Extraction des frames
frame_count = 0
saved_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_name = f"Frames-Fish/frame_{frame_count:04d}.jpg"
    cv2.imwrite(frame_name, frame)
    saved_count += 1
    frame_count += 1

cap.release()
print(f"{saved_count} frames extraites dans le dossier 'Frames-Fish/'")

# 7. Visualiser chaque 10e frame
import glob

frame_files = sorted(glob.glob("Frames-Fish/*.jpg"))
print(f"Nombre total de frames : {len(frame_files)}")


250 frames extraites dans le dossier 'Frames-Fish/'
Nombre total de frames : 250


In [5]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [6]:
pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 2.3 MB/s eta 0:00:00


In [7]:
pip install ultralytics

In [8]:
import cv2
import csv
import os
import math
from ultralytics import YOLO


# PARAMÈTRES

VIDEO_PATH = "Birds-2 (1).mp4"
OUTPUT_VIDEO = "tracked_birds.mp4"
OUTPUT_CSV = "birds_tracking.csv"
TEMP_DIR = "temp_frames"

os.makedirs(TEMP_DIR, exist_ok=True)

CONF_THRESH = 0.45
IOU_THRESH = 0.5
MAX_DISTANCE = 60

RED = (0, 0, 255)


# YOLO MODEL
model = YOLO("yolov8m.pt")

# OUTILS

def center(x1, y1, x2, y2):
    return ((x1 + x2)//2, (y1 + y2)//2)

def dist(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def draw_cross(img, x, y, s=8):
    cv2.line(img, (x-s, y-s), (x+s, y+s), RED, 2)
    cv2.line(img, (x+s, y-s), (x-s, y+s), RED, 2)

def is_duplicate(box, boxes, threshold=25):
    x1,y1,x2,y2 = box
    c1 = ((x1+x2)//2, (y1+y2)//2)

    for b in boxes:
        a1,b1,a2,b2 = b
        c2 = ((a1+a2)//2, (b1+b2)//2)

        d = dist(c1, c2)
        if d < threshold:
            return True
    return False
# VIDEO
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Erreur vidéo")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# TRACKING
tracks = {}
next_id = 0

csv_rows = []
frame_id = 0

print(" Début détection YOLO Bird Strike...")
# LOOP
while True:

    ret, frame = cap.read()
    if not ret:
        break

    # YOLO DETECTION + NMS

    results = model(
        frame,
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        verbose=False
    )

    detections = []

    for r in results:
        for box in r.boxes:

            cls = int(box.cls[0])
            name = model.names[cls]
            conf = float(box.conf[0])

            if name == "bird" and conf >= CONF_THRESH:

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                detections.append((x1, y1, x2, y2))

    # SUPPRESSION DOUBLONS SPATIAUX
    filtered = []

    for d in detections:
        if not is_duplicate(d, filtered, threshold=25):
            filtered.append(d)

    detections = filtered
    # TRACKING SIMPLE
    new_tracks = {}
    used = set()

    # match anciens tracks
    for tid, old_center in tracks.items():

        best = -1
        best_d = MAX_DISTANCE

        for i, d in enumerate(detections):

            if i in used:
                continue

            c = center(*d)
            distance_val = dist(old_center, c)

            if distance_val < best_d:
                best_d = distance_val
                best = i

        if best != -1:
            used.add(best)
            new_tracks[tid] = center(*detections[best])

    # nouveaux oiseaux
    for i, d in enumerate(detections):

        if i not in used:
            new_tracks[next_id] = center(*d)
            next_id += 1

    tracks = new_tracks
    # DRAW + CSV
    for i, d in enumerate(detections):

        x1, y1, x2, y2 = d

        cx, cy = center(x1, y1, x2, y2)

        radius = max((x2-x1)//2, (y2-y1)//2)

        cv2.circle(frame, (cx, cy), radius, RED, 2)
        draw_cross(frame, cx, cy)

        csv_rows.append([frame_id, cx, cy, x1, y1, x2, y2])

    cv2.putText(
        frame,
        f"Birds: {len(detections)} Frame:{frame_id}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,255),
        2
    )

    cv2.imwrite(f"{TEMP_DIR}/frame_{frame_id:06d}.jpg", frame)

    if frame_id % 20 == 0:
        print(f"Frame {frame_id}/{total_frames}")

    frame_id += 1

cap.release()
# VIDEO OUTPUT
print("Création vidéo...")

os.system(
    f"ffmpeg -y -framerate {int(fps)} "
    f"-i {TEMP_DIR}/frame_%06d.jpg "
    f"-c:v libx264 -pix_fmt yuv420p {OUTPUT_VIDEO}"
)
# CSV
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "cx", "cy", "x1", "y1", "x2", "y2"])
    writer.writerows(csv_rows)

print("FIN")
print("Vidéo :", OUTPUT_VIDEO)
print("CSV :", OUTPUT_CSV)
print("Total oiseaux :", next_id)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Début détection YOLO Bird Strike...
Frame 0/250
Frame 20/250
Frame 40/250
Frame 60/250
Frame 80/250
Frame 100/250
Frame 120/250
Frame 140/250
Frame 160/250
Frame 180/250
Frame 200/250
Frame 220/250
Frame 240/250
Création vidéo...
FIN
Vidéo : tracked_birds.mp4
CSV : birds_tracking.csv
Total oiseaux : 115
